# Fine-Tuning du Tuteur d'Anglais Oral avec Unsloth (LoRA)

Ce notebook entraîne nos deux déclinaisons de tuteur oral selon le matériel cible :
- **Profil Standard (PC & Smartphone 6Go+)** : `Qwen2.5-1.5B-Instruct` (~1,1 Go RAM)
- **Profil Éco / Léger (Smartphone 4Go)** : `Qwen2.5-0.5B-Instruct` (~350 Mo RAM)

- **Durée estimée** : ~8-12 minutes par modèle sur GPU T4 gratuit Colab.
- **Sortie** : Fichiers quantifiés `.gguf` prêts pour Ollama / mobile.

### 1. Installation des dépendances Unsloth (ultra-rapide)

In [ ]:
# Installation directe recommandée d'Unsloth (compatible Python 3.12/3.13)
!pip install unsloth

### 2. Chargement du modèle de base en 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

# Choisis le profil cible en décommentant la ligne souhaitée :
# ------------------------------------------------------------
# Profil Standard (PC / Mobile 6Go+) :
model_name = "unsloth/Qwen2.5-1.5B-Instruct"

# Profil Éco (Mobile 4Go de RAM) :
# model_name = "unsloth/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Configuration LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 4. Chargement du dataset `tutor_finetune.jsonl`
> Un bouton d'upload apparaîtra automatiquement pour choisir le fichier depuis ton ordinateur si besoin.

In [ ]:
import os
from google.colab import files
from datasets import load_dataset

# Ouvre une fenêtre d'explorateur pour choisir le fichier s'il n'est pas déjà présent
dataset_filename = "tutor_finetune.jsonl"
if not os.path.exists(dataset_filename):
    print("Sélectionne ton fichier tutor_finetune.jsonl depuis ton ordinateur :")
    uploaded = files.upload()

dataset = load_dataset("json", data_files=dataset_filename, split="train")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

### 5. Entraînement du modèle

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 6. Exportation directe au format GGUF (pour Ollama)

In [ ]:
# Sauvegarde en format GGUF quantifié Q4_K_M
model.save_pretrained_gguf("tutor_model_gguf", tokenizer, quantization_method = "q4_k_m")

print("Export GGUF terminé avec succès ! Tu peux télécharger le fichier pour Ollama.")